In [1]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [16]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import numpy as np
from tqdm import tqdm
from FlagEmbedding import FlagReranker

In [59]:
# embedding_model = SentenceTransformer('BAAI/bge-m3', device='cpu', backend='onnx')
embedding_model = SentenceTransformer(
    r'C:\Users\ThePlayer\.cache\huggingface\hub\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181', 
    device='cpu', local_files_only=True)

reranker = FlagReranker(
    r'C:\Users\ThePlayer\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3\snapshots\953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e', 
    device='cpu', use_fp16=True, local_files_only=True) 


In [54]:
client = chromadb.PersistentClient(path="./database/chroma_1")
collection = client.get_collection(name="danbooru_tags")


In [55]:
query_text = "华丽的女性服饰"

recall_output = 100 # 输出太多 reranker 会有压力
rerank_output = 20 # 输出太多 

In [56]:
query_embedding = embedding_model.encode(query_text, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=recall_output
)

print("检索到的标签:", results['documents'])

检索到的标签: [['ornate_clothes', 'multicolored_dress', 'clothed_female_nude_female', 'multicolored_clothes', 'layered_dress', 'glowing_clothes', 'pleated_dress', 'checkered_dress', 'multi-strapped_dress', 'layered_clothes', 'armored_dress', 'hooded_dress', 'high-waist_skirt', 'shiny_clothes', "women's_wallet", 'holding_dress', 'vintage_clothes', 'expressive_clothes', 'clothes', 'traditional_dress', 'highleg_dress', 'white_dress', 'embroidered_dress', 'vertical-striped_dress', 'dress', 'revealing_clothes', 'undressing', 'two-sided_dress', 'long_dress', 'bridal_lingerie', 'clothes_around_waist', 'multicolored_skirt', 'embellished_costume', 'tall_female', 'floral_dress', 'traditional_clothes', 'weighted_clothes', 'dress_straps', 'painted_clothes', 'fur-trimmed_dress', 'checkered_clothes', 'lingerie', 'two-tone_dress', 'changing_clothes', 'camouflage_dress', 'skirt_rolled_up', 'pleated_skirt', 'striped_dress', 'clothes_in_front', 'collared_dress', 'renaissance_clothes', 'dress_pants', 'high-wai

In [57]:
candidates = results['documents'][0]
candidates = [p.replace("_", " ") for p in candidates]

# ",".join(candidates[:20])
candidates[:5]


['ornate clothes',
 'multicolored dress',
 'clothed female nude female',
 'multicolored clothes',
 'layered dress']

In [60]:
pairs = [[query_text, candidate] for candidate in candidates]

# 计算得分 (分数越高越相关)
scores = reranker.compute_score(pairs)

# 将得分与候选标签组合并排序
reranked_results = sorted(
    zip(candidates, scores), 
    key=lambda x: x[1], 
    reverse=True
)

# 输出前 20 个最精准的结果
for tag, score in reranked_results[:rerank_output]:
    print(f"Tag: {tag}, Score: {score:.4f}")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Tag: ornate clothes, Score: 6.0869
Tag: glowing clothes, Score: 4.4276
Tag: embellished costume, Score: 4.3810
Tag: shiny clothes, Score: 4.3571
Tag: white dress, Score: 2.4230
Tag: expressive clothes, Score: 2.0793
Tag: revealing clothes, Score: 1.9186
Tag: weighted clothes, Score: 1.3787
Tag: vintage clothes, Score: 1.1265
Tag: wringing dress, Score: 1.0621
Tag: pleated dress, Score: 0.9675
Tag: wardrobe, Score: 0.9623
Tag: dress suit, Score: 0.8534
Tag: oversized clothes, Score: 0.7401
Tag: renaissance clothes, Score: 0.7132
Tag: clothes, Score: 0.4744
Tag: dress, Score: 0.4419
Tag: dressing, Score: 0.3275
Tag: checkered clothes, Score: 0.0005
Tag: long dress, Score: -0.0559
